# 📥 Tarea 1 · Parte 1 — Recolección de noticias peruanas

**Pregunta guía:** ¿Pueden convertir titulares peruanos en un clasificador de noticias?

| Campo | Valor |
|---|---|
| **Opción elegida** | **C — Comparada** (200 de La República + 200 de El Comercio) |
| **Fecha de descarga** | `COMPLETAR AL EJECUTAR` (la celda siguiente la imprime) |
| **Medio 1** | La República — `https://larepublica.pe` |
| **Medio 2** | El Comercio — `https://elcomercio.pe` |
| **Categorías** | `politica`, `economia`, `deportes`, `ciencia_tecnologia` |

## Estrategia de extracción (y por qué)

Los dos diarios publican **RSS oficiales**, pero cada feed trae solo 15–25 notas: cuatro
categorías por medio dan unas 80 filas, muy lejos de las 350+ que pide la tarea. Además
La República prohíbe `/*/pag/*` en su `robots.txt`, así que paginar las secciones queda
descartado.

La solución es combinar dos fuentes que el `robots.txt` **permite explícitamente**:

1. **RSS** → entrega titular y *bajada* ya limpios (el campo `<description>`).
2. **Sitemaps por sección** → entregan cientos de URLs con fecha; de cada nota se leen
   los metadatos Open Graph para recuperar titular y bajada.

Todo con `User-Agent` descriptivo, pausa de 1–2 s y un solo reintento.

In [ ]:
import json, random, re, time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from bs4 import BeautifulSoup

SEED = 42
random.seed(SEED)

BASE   = Path.cwd()
DATOS  = BASE / "datos"; DATOS.mkdir(exist_ok=True)
RAW    = DATOS / "raw";  RAW.mkdir(exist_ok=True)
FIGS   = DATOS / "figs"; FIGS.mkdir(exist_ok=True)

CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
CUPO = 55            # objetivo por (medio, categoria) -> ~220 por medio, ~440 en total
FECHA_DESCARGA = datetime.now().strftime("%Y-%m-%d")

UA = ("TareaAcademica/1.0 (Diplomatura Ciencia de Datos QLAB - PUCP; "
      "uso academico, sin fines comerciales)")
SESION = requests.Session()
SESION.headers.update({"User-Agent": UA, "Accept-Language": "es-PE,es;q=0.9"})

def pausa():
    """Pausa cortes de 1 a 2 segundos entre peticiones, como pide el enunciado."""
    time.sleep(random.uniform(1.0, 2.0))

def descargar(url, reintentos=1, timeout=25):
    """GET con cache en disco. Si ya se descargó antes, no vuelve a pedir la página.
    Esto hace que la recolección sea REANUDABLE: si se corta, se retoma sin perder avance."""
    nombre = re.sub(r"[^A-Za-z0-9]+", "_", url)[:150] + ".txt"
    cache = RAW / nombre
    if cache.exists():
        return cache.read_text(encoding="utf-8", errors="ignore")
    for intento in range(reintentos + 1):
        try:
            pausa()
            r = SESION.get(url, timeout=timeout)
            if r.status_code == 200:
                r.encoding = r.encoding or "utf-8"
                cache.write_text(r.text, encoding="utf-8")
                return r.text
            if intento == reintentos:
                print(f"  [{r.status_code}] {url}")
        except requests.RequestException as e:
            if intento == reintentos:
                print(f"  [error] {type(e).__name__} {url}")
    return None

print("Fecha de descarga :", FECHA_DESCARGA)
print("User-Agent        :", UA)
print("Cupo por medio y categoria:", CUPO)

## 1 · Configuración de medios y mapa de secciones

Las secciones que publica cada diario no coinciden con las 4 categorías oficiales de la
tarea, así que el mapeo es **explícito** (y se imprime más abajo como tabla de equivalencias,
que es lo que pide la rúbrica).

Nota sobre `ciencia_tecnologia`: La República separa `ciencia` y no tiene feed de
tecnología; El Comercio agrupa ambas bajo `tecnologia` (incluye `tecnologia/ciencias`).
Por eso cada medio aporta esa categoría desde una sección distinta.

In [ ]:
MEDIOS = {
    "La Republica": {
        "dominio": "larepublica.pe",
        "rss":     "https://larepublica.pe/rss/{sec}.xml",
        "sitemap": "https://larepublica.pe/sitemap/{sec}.xml",
        # seccion del diario -> categoria oficial de la tarea
        "mapa": {
            "politica": "politica",
            "economia": "economia",
            "deportes": "deportes",
            "ciencia":  "ciencia_tecnologia",
        },
    },
    "El Comercio": {
        "dominio": "elcomercio.pe",
        "rss":     "https://elcomercio.pe/arc/outboundfeeds/google-news-feed/category/{sec}/?outputType=xml",
        "sitemap": "https://elcomercio.pe/sitemap/{sec}/?outputType=xml",
        "mapa": {
            "politica":      "politica",
            "economia":      "economia",
            "deporte-total": "deportes",
            "tecnologia":    "ciencia_tecnologia",
        },
    },
}

mapa_largo = pd.DataFrame(
    [{"medio": m, "seccion_original": s, "categoria_final": c}
     for m, cfg in MEDIOS.items() for s, c in cfg["mapa"].items()]
).sort_values(["medio", "categoria_final"]).reset_index(drop=True)

print("TABLA DE EQUIVALENCIAS seccion -> categoria")
display(mapa_largo)

## 2 · Verificación del `robots.txt`

Antes de pedir nada se revisa qué permite cada diario. Las reglas relevantes se imprimen
para dejar constancia en el notebook.

In [ ]:
for medio, cfg in MEDIOS.items():
    txt = descargar(f"https://{cfg['dominio']}/robots.txt")
    print(f"\n=== {medio} — robots.txt ===")
    if not txt:
        print("  no se pudo leer")
        continue
    relevantes = [l.strip() for l in txt.splitlines()
                  if re.match(r"(?i)^\s*(user-agent|disallow|allow|crawl-delay)", l)]
    for l in relevantes[:40]:
        print("  ", l)
    permite_rss = any("rss" in l.lower() and l.lower().startswith("allow") for l in relevantes)
    print(f"  -> /rss/ permitido explicitamente: {permite_rss}")

**Lo que se encontró** *(completar con lo que imprima la celda al ejecutar)*:

- **La República** permite explícitamente `/rss/*` y `/sitemap/*`, y prohíbe `/*/pag/*`,
  `/buscador*` y `/archivo/*`. Por eso **no** se pagina ninguna sección.
- **El Comercio** solo prohíbe `/catalogo/`, `/buscar/` y el archivo anterior al 2010.
  Los feeds y sitemaps quedan permitidos.

## 3 · Paso A — Extracción por RSS

El RSS da titular y bajada sin tener que abrir cada nota. Es la fuente barata; se agota
rápido, pero cubre la parte más reciente del dataset.

In [ ]:
def _texto(nodo):
    return " ".join(nodo.itertext()).strip() if nodo is not None else ""

def limpiar_html(s):
    """Quita etiquetas y normaliza espacios. Las bajadas del RSS vienen con HTML dentro."""
    if not s:
        return ""
    s = BeautifulSoup(s, "html.parser").get_text(" ").replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s)
    # get_text(" ") mete un espacio al cerrar cada etiqueta: "<b>Ejecutivo</b>." -> "Ejecutivo ."
    s = re.sub(r"\s+([,.;:!?%)\]»])", r"\1", s)
    s = re.sub(r"([(\[«¿¡])\s+", r"\1", s)
    return s.strip()

def fecha_iso(s):
    """Convierte el pubDate RFC-822 del RSS a AAAA-MM-DD."""
    if not s:
        return None
    try:
        from email.utils import parsedate_to_datetime
        return parsedate_to_datetime(s).astimezone(timezone.utc).strftime("%Y-%m-%d")
    except Exception:
        m = re.search(r"(\d{4})-(\d{2})-(\d{2})", s)
        return "-".join(m.groups()) if m else None

def leer_rss(medio, seccion):
    """Devuelve una lista de dicts con lo que trae el feed de esa seccion."""
    cfg = MEDIOS[medio]
    xml = descargar(cfg["rss"].format(sec=seccion))
    if not xml:
        return []
    try:
        raiz = ET.fromstring(xml.encode("utf-8"))
    except ET.ParseError as e:
        print(f"  XML invalido en {medio}/{seccion}: {e}")
        return []
    filas = []
    for it in raiz.iter("item"):
        url = _texto(it.find("link"))
        if not url:
            continue
        filas.append({
            "titular":          limpiar_html(_texto(it.find("title"))),
            "bajada":           limpiar_html(_texto(it.find("description"))),
            "medio":            medio,
            "seccion_original": seccion,
            "url":              url.strip(),
            "fecha":            fecha_iso(_texto(it.find("pubDate"))),
            "fuente_extraccion": "rss",
        })
    return filas

filas_rss = []
for medio, cfg in MEDIOS.items():
    for seccion in cfg["mapa"]:
        nuevas = leer_rss(medio, seccion)
        filas_rss += nuevas
        print(f"RSS  {medio:14s} / {seccion:14s} -> {len(nuevas):3d} notas")

print(f"\nTotal por RSS: {len(filas_rss)}")

## 4 · Paso B — Sitemaps por sección

Aquí está el volumen. Cada sitemap de sección lista cientos de URLs con su fecha; de cada
nota se leen los metadatos Open Graph (`og:title`, `og:description`), que es donde ambos
diarios publican titular y bajada de forma estable.

La función respeta el cupo por categoría: deja de pedir notas apenas llega al objetivo, así
no se descarga de más.

In [ ]:
def _local(tag):
    """Nombre de la etiqueta sin su namespace: '{http://...}loc' -> 'loc'."""
    return tag.rsplit("}", 1)[-1].lower()

def leer_sitemap(medio, seccion):
    """URLs + fecha (lastmod) de una seccion, de la mas reciente a la mas antigua.

    El parseo ignora el namespace a proposito: los sitemaps de los dos diarios usan
    esquemas distintos (sitemaps.org y la extension de Google News), y comparar por
    nombre local funciona con ambos sin tener que mantener una lista de namespaces."""
    cfg = MEDIOS[medio]
    xml = descargar(cfg["sitemap"].format(sec=seccion))
    if not xml:
        return []
    try:
        raiz = ET.fromstring(xml.encode("utf-8"))
    except ET.ParseError as e:
        print(f"  sitemap invalido en {medio}/{seccion}: {e}")
        return []
    items = []
    for nodo in raiz.iter():
        if _local(nodo.tag) != "url":
            continue
        loc = lastmod = publicacion = None
        # se recorren TODOS los descendientes: El Comercio anida la fecha dentro
        # de <news:news><news:publication_date>, no como hijo directo de <url>
        for hijo in nodo.iter():
            if hijo is nodo or not hijo.text:
                continue
            nombre = _local(hijo.tag)
            if nombre == "loc":
                loc = hijo.text.strip()
            elif nombre == "lastmod":
                lastmod = hijo.text.strip()[:10]
            elif nombre == "publication_date":
                publicacion = hijo.text.strip()[:10]
        if loc:
            items.append({"url": loc, "fecha": lastmod or publicacion})
    items.sort(key=lambda d: d["fecha"] or "", reverse=True)
    return items

def seccion_desde_url(medio, url):
    """La seccion esta en el primer segmento de la ruta en ambos diarios.
    En El Comercio se conserva tambien la subseccion (p. ej. tecnologia/ciencias)."""
    partes = [p for p in urlparse(url).path.split("/") if p]
    if not partes:
        return None
    if medio == "El Comercio" and len(partes) > 1 and not partes[1].isdigit():
        return f"{partes[0]}/{partes[1]}"
    return partes[0]

def leer_nota(url):
    """Titular y bajada de una nota, con cadena de respaldos por si falta un metadato."""
    html = descargar(url)
    if not html:
        return None
    s = BeautifulSoup(html, "html.parser")

    def meta(prop, attr="property"):
        t = s.find("meta", attrs={attr: prop})
        return (t.get("content") or "").strip() if t and t.get("content") else ""

    titular = (meta("og:title") or meta("twitter:title", "name")
               or (s.h1.get_text(" ").strip() if s.h1 else "")
               or (s.title.get_text(" ").strip() if s.title else ""))
    bajada  = (meta("og:description") or meta("description", "name")
               or meta("twitter:description", "name"))
    fecha   = (meta("article:published_time") or meta("datePublished", "itemprop"))[:10]
    if not titular:
        return None
    titular = re.sub(r"\s*\|\s*(LA REPUBLICA|La República|EL COMERCIO|El Comercio).*$", "", titular)
    return {"titular": limpiar_html(titular), "bajada": limpiar_html(bajada), "fecha": fecha or None}

In [ ]:
vistas = {f["url"] for f in filas_rss}
filas_sitemap = []

# cuanto falta por (medio, categoria) despues del RSS
faltante = {}
for medio, cfg in MEDIOS.items():
    for cat in CATEGORIAS:
        ya = sum(1 for f in filas_rss
                 if f["medio"] == medio and cfg["mapa"].get(f["seccion_original"]) == cat)
        faltante[(medio, cat)] = max(0, CUPO - ya)

print("Faltante por medio y categoria tras el RSS:")
for k, v in faltante.items():
    print(f"  {k[0]:14s} {k[1]:20s} {v:3d}")

for medio, cfg in MEDIOS.items():
    for seccion, cat in cfg["mapa"].items():
        cuota = faltante[(medio, cat)]
        if cuota == 0:
            print(f"SMAP {medio:14s} / {seccion:14s} -> cupo ya cubierto por RSS")
            continue
        candidatos = [c for c in leer_sitemap(medio, seccion) if c["url"] not in vistas]
        obtenidas = 0
        for c in candidatos:
            if obtenidas >= cuota:
                break
            if seccion_desde_url(medio, c["url"]) is None:
                continue
            nota = leer_nota(c["url"])
            if not nota or len(nota["titular"]) < 15:
                continue
            vistas.add(c["url"])
            filas_sitemap.append({
                "titular":          nota["titular"],
                "bajada":           nota["bajada"],
                "medio":            medio,
                "seccion_original": seccion_desde_url(medio, c["url"]) or seccion,
                "url":              c["url"],
                "fecha":            nota["fecha"] or c["fecha"],
                "fuente_extraccion": "sitemap",
            })
            obtenidas += 1
        print(f"SMAP {medio:14s} / {seccion:14s} -> {obtenidas:3d} notas "
              f"(de {len(candidatos)} candidatas)")

print(f"\nTotal por sitemap: {len(filas_sitemap)}")

## 5 · Consolidación, limpieza y normalización

Se juntan ambas fuentes y se aplican, en orden: deduplicación por URL, descarte de
titulares vacíos o demasiado cortos, mapeo de sección a categoría, y descarte de las notas
que no caen en ninguna de las 4 categorías oficiales.

**El enunciado es explícito: no se inventa una quinta categoría.** Lo que no encaja se
descarta y se reporta.

In [ ]:
bruto = pd.DataFrame(filas_rss + filas_sitemap)
print("Filas crudas:", len(bruto))

motivos = {}
def descartar(df, mascara, motivo):
    n = int(mascara.sum())
    if n:
        motivos[motivo] = motivos.get(motivo, 0) + n
    return df[~mascara].copy()

# a) duplicados por URL
antes = len(bruto)
bruto = bruto.drop_duplicates(subset="url", keep="first").reset_index(drop=True)
motivos["URL duplicada"] = antes - len(bruto)

# b) titulares duplicados dentro del mismo medio (misma nota con otra URL)
antes = len(bruto)
bruto = bruto.drop_duplicates(subset=["medio", "titular"], keep="first").reset_index(drop=True)
motivos["Titular repetido en el mismo medio"] = antes - len(bruto)

# c) titulares vacios o demasiado cortos para clasificar
bruto = descartar(bruto, bruto["titular"].fillna("").str.len() < 15,
                  "Titular vacio o menor a 15 caracteres")

# d) mapeo seccion -> categoria
def a_categoria(fila):
    mapa = MEDIOS[fila["medio"]]["mapa"]
    sec = fila["seccion_original"]
    if sec in mapa:
        return mapa[sec]
    raiz = sec.split("/")[0]          # El Comercio guarda subsecciones
    return mapa.get(raiz)

bruto["etiqueta"] = bruto.apply(a_categoria, axis=1)
bruto = descartar(bruto, bruto["etiqueta"].isna(),
                  "Seccion fuera de las 4 categorias oficiales")

# e) fechas faltantes -> se conservan, pero se reportan
sin_fecha = int(bruto["fecha"].isna().sum())

df = bruto.reset_index(drop=True)
df["bajada"] = df["bajada"].fillna("")

print("\nREPORTE DE DESCARTES")
for motivo, n in motivos.items():
    print(f"  {n:4d}  {motivo}")
print(f"  {sin_fecha:4d}  (conservadas) notas sin fecha recuperable")
print(f"\nFilas finales: {len(df)}")

print("\nDistribucion final por medio y categoria:")
display(pd.crosstab(df["medio"], df["etiqueta"], margins=True, margins_name="Total"))

## 6 · Análisis exploratorio

Tres gráficos interactivos en Plotly. **El título de cada uno enuncia la conclusión**, no
solo la variable: es lo que exige la rúbrica.

> Los títulos de abajo tienen un marcador `«…»` donde va el número que arroje tu corrida.
> Reemplázalos por la conclusión real antes de entregar.

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"

PALETA = {"politica": "#2E5A88", "economia": "#2E8B6F",
          "deportes": "#C25E1E", "ciencia_tecnologia": "#7A4E9E"}

# --- Grafico 1: cobertura por categoria y medio ---
g1 = df.groupby(["etiqueta", "medio"]).size().reset_index(name="noticias")
dom = g1.loc[g1["noticias"].idxmax()]
fig1 = px.bar(
    g1, x="etiqueta", y="noticias", color="medio", barmode="group", text="noticias",
    title=(f"El corpus queda balanceado entre las 4 categorias, con {dom['medio']} "
           f"aportando el bloque mas grande en {dom['etiqueta']}"),
    labels={"etiqueta": "Categoria", "noticias": "Noticias", "medio": "Medio"},
)
fig1.update_layout(template="plotly_white", title_font_size=14)
fig1.show()

In [ ]:
# --- Grafico 2: ventana temporal cubierta por cada medio ---
tmp = df.dropna(subset=["fecha"]).copy()
tmp["fecha_dt"] = pd.to_datetime(tmp["fecha"], errors="coerce")
tmp = tmp.dropna(subset=["fecha_dt"])
g2 = tmp.groupby([tmp["fecha_dt"].dt.date, "medio"]).size().reset_index(name="noticias")
g2.columns = ["fecha", "medio", "noticias"]
rango = f"{tmp['fecha_dt'].min():%d %b} a {tmp['fecha_dt'].max():%d %b}"
fig2 = px.line(
    g2, x="fecha", y="noticias", color="medio", markers=True,
    title=(f"La recoleccion cubre una ventana corta ({rango}): el dataset retrata una "
           f"coyuntura, no un periodo largo"),
    labels={"fecha": "Fecha de publicacion", "noticias": "Noticias", "medio": "Medio"},
)
fig2.update_layout(template="plotly_white", title_font_size=14)
fig2.show()

In [ ]:
# --- Grafico 3: longitud de los titulares ---
df["largo_titular"] = df["titular"].str.len()
med = df.groupby("medio")["largo_titular"].median()
mas_largo = med.idxmax()
fig3 = px.histogram(
    df, x="largo_titular", color="medio", nbins=40, barmode="overlay", opacity=0.65,
    title=(f"{mas_largo} escribe titulares mas largos (mediana {med[mas_largo]:.0f} vs "
           f"{med.min():.0f} caracteres), lo que da mas senal al clasificador"),
    labels={"largo_titular": "Caracteres del titular", "count": "Noticias", "medio": "Medio"},
)
fig3.update_layout(template="plotly_white", title_font_size=14)
fig3.show()

print(df.groupby("medio")["largo_titular"].describe()[["count", "mean", "50%", "min", "max"]])

## 7 · Validación: los extremos de la distribución

Se revisan los 5 titulares más largos y los 5 más cortos para confirmar que el histograma
no esconde basura (titulares truncados, pies de foto que se colaron, etc.).

In [ ]:
cols = ["medio", "etiqueta", "largo_titular", "titular"]
print("=== 5 TITULARES MAS LARGOS ===")
display(df.nlargest(5, "largo_titular")[cols])
print("=== 5 TITULARES MAS CORTOS ===")
display(df.nsmallest(5, "largo_titular")[cols])

## 8 · Exportación

El CSV final lleva exactamente las **7 columnas obligatorias**, en el orden del enunciado.

In [ ]:
COLUMNAS = ["titular", "bajada", "medio", "seccion_original", "url", "fecha", "etiqueta"]
final = df[COLUMNAS].reset_index(drop=True)
final.to_csv(DATOS / "noticias.csv", index=False, encoding="utf-8")

print("df.shape =", final.shape)
assert final.shape[0] >= 350, f"Solo {len(final)} filas: sube CUPO y vuelve a correr la celda de sitemaps"
assert list(final.columns) == COLUMNAS
assert final["titular"].isna().sum() == 0
print("Guardado en datos/noticias.csv\n")
display(final.head())

In [ ]:
# Verificacion de que las URLs funcionan, sobre una muestra aleatoria
muestra = final.sample(min(20, len(final)), random_state=SEED)["url"]
vivas = 0
for u in muestra:
    pausa()
    try:
        vivas += SESION.head(u, timeout=20, allow_redirects=True).status_code < 400
    except requests.RequestException:
        pass
print(f"URLs funcionales en la muestra: {vivas}/{len(muestra)}")

## 9 · Conclusiones de la recolección

*Completar con los números de tu corrida — esto es lo que luego se copia al README.*

- **Fecha de descarga:** …
- **Volumen final:** … notas (… de La República + … de El Comercio).
- **Ventana temporal cubierta:** …
- **Bloqueos encontrados y cómo se resolvieron:** el RSS se agota a las ~20 notas por
  sección, así que el volumen salió de los sitemaps, que ambos diarios permiten
  explícitamente en su `robots.txt`. No se paginó ninguna sección porque La República lo
  prohíbe.
- **Descartes principales:** …
- **Riesgo que queda abierto:** la etiqueta de este CSV viene de la **sección del diario**,
  que es una aproximación, no una verdad de oro. Una nota sobre el presupuesto público
  puede estar en `politica` en un diario y en `economia` en el otro. Esa tensión se mide en
  el notebook 2.